In [ ]:
import pandas as pd

In [ ]:
envisoft_path = "/home/slow_data/Air_Quality/filtered_envisoft_air_quality_weather_data.csv"
iqair_path = "/home/slow_data/Air_Quality/IQAir_air_quality.csv"

envisoft_output_path = "/home/work1/projects/Air_Quality/Masterdata/envisoft_stations.csv"
iqair_output_path = "/home/work1/projects/Air_Quality/Masterdata/iqair_stations.csv"

In [ ]:
envisoft_df = pd.read_csv(envisoft_path)
iqair_df = pd.read_csv(iqair_path)

In [ ]:
envisoft_df

In [ ]:
envisoft_station_df = (
    envisoft_df[['ID', 'Name', 'Latitude', 'Longitude']]
    .drop_duplicates(subset='ID')
)
envisoft_station_df = envisoft_station_df.reset_index(drop=True).sort_values('Name')

In [ ]:
import re

# Mapping tên tỉnh/thành → region
REGION_MAP = {
    # Bắc
    'Bắc Giang': 'Bắc',
    'Hà Nam': 'Bắc',
    'Hà Nội': 'Bắc',
    'Phú Thọ': 'Bắc',
    'Thái Nguyên': 'Bắc',
    # Trung
    'Quảng Bình': 'Trung',
    'Đà Nẵng': 'Trung',
    # Nam
    'Bình Dương': 'Nam',
    'HCM': 'Nam',
    'Long An': 'Nam',
}

def normalize_name(name: str) -> str:
    """
    Chuẩn hóa tên trạm:
    - Loại bỏ '(KK)' ở cuối
    - Chuẩn hóa viết hoa đầu từ
    - Trim khoảng trắng thừa
    """
    # Bỏ hậu tố (KK)
    name = re.sub(r'\s*\(KK\)\s*$', '', name).strip()

    # Chuẩn hóa TP./P./Đ. (giữ nguyên viết tắt, chỉ strip khoảng trắng)
    name = re.sub(r'\s+', ' ', name)

    return name

def extract_province(name: str) -> str:
    """Trích tên tỉnh/thành từ đầu tên trạm (trước dấu ':')."""
    if ':' in name:
        province = name.split(':')[0].strip()
        # Chuẩn hóa 'Thái nguyên' → 'Thái Nguyên'
        province = province.title()
        # Giữ lại 'HCM' không title-case
        if province.upper() in ('HCM', 'TP.HCM'):
            province = 'HCM'
        return province
    return ''

def get_region(name: str) -> str:
    province = extract_province(name)
    return REGION_MAP.get(province, 'Không xác định')

# Áp dụng
envisoft_station_df['Name'] = envisoft_station_df['Name'].apply(normalize_name)
envisoft_station_df['Region'] = envisoft_station_df['Name'].apply(get_region)
envisoft_station_df = envisoft_station_df.sort_values(by=["Region", "Name"])

In [ ]:
envisoft_station_df.rename(columns={'Name': 'station_name', "Latitude": "latitude", "Longitude": "Longitude", "Region": "region"}, inplace=True)

In [ ]:
envisoft_station_df

In [ ]:
iqair_df

In [ ]:
cols = ['station_name', 'longitude', 'latitude']

iqair_station_df = (
    iqair_df[cols]
    .drop_duplicates(subset='station_name')
)

# Round tọa độ để gom các điểm gần nhau (sai số ~100m)
iqair_station_df = iqair_station_df.copy()
iqair_station_df['_lat_r'] = iqair_station_df['latitude'].round(3)
iqair_station_df['_lon_r'] = iqair_station_df['longitude'].round(3)

iqair_station_df = iqair_station_df.sort_values(
    'station_name', key=lambda s: s.str.len(), ascending=False
)
iqair_station_df = (
    iqair_station_df
    .drop_duplicates(subset=['_lat_r', '_lon_r'])
    .drop(columns=['_lat_r', '_lon_r'])
    .reset_index(drop=True)
)

In [ ]:
iqair_station_df = iqair_station_df[
    iqair_station_df['station_name'] != 'Office IQAir Ha Noi'
].reset_index(drop=True)

In [ ]:
iqair_station_df

In [ ]:
REGION_MAP = {
    # Bắc
    'Hà Nội':        'Bắc',
    'Thái Nguyên':   'Bắc',
    'Thái Bình':     'Bắc',
    'Hải Dương':     'Bắc',
    # Trung
    'Quảng Bình':    'Trung',
    'Thừa Thiên Huế':'Trung',
    # Nam
    'Vũng Tàu':      'Nam',
    'Trà Vinh':      'Nam',
}

# Map thủ công cho các tên không có prefix "Tỉnh:"
MANUAL_MAP = {
    'Minh Khai - Bắc Từ Liêm':       'Bắc',
    'IQAir Ha Noi':                   'Bắc',
    'HCM - FPT Thuduc':               'Nam',
    'IQAir Vietnam - Saigon Pearl':   'Nam',
}

def get_region(name: str) -> str:
    if name in MANUAL_MAP:
        return MANUAL_MAP[name]
    if ':' in name:
        province = name.split(':')[0].strip().title()
        return REGION_MAP.get(province, 'Không xác định')
    return 'Không xác định'

iqair_station_df['region'] = iqair_station_df['station_name'].apply(get_region)
iqair_station_df = iqair_station_df.sort_values(by=["region", "station_name"])

In [ ]:
iqair_station_df

In [ ]:
iqair_station_df.to_csv(iqair_output_path, index=False)
envisoft_station_df.to_csv(envisoft_output_path, index=False)